# Nvidia Trailed the Market by 18 Points the Week Huang Resigned. Revenue Is Still Up 106%.

**The market rose 3% the week Nvidia fell 11%. At $166 and 21 times earnings, the discount is real, and so is the risk. Is it a buy before Tuesday's earnings?**

Jensen Huang [resigned as Nvidia's CEO on Thursday](https://www.cnbc.com/2026/11/12/nvidia-nvda-jensen-huang-second-affair.html), 33 years after he co-founded it, following a week of reports about his personal relationships, including one with an employee who reported to him. CFO Colette Kress is interim CEO. There is no permanent successor.

The stock [fell as much as 14% Thursday morning](https://www.nbcnews.com/business/markets/nvidia-stock-huang-resigns-rcna608958), then closed at **$166.19, down 4.3%**. It sits 31% below its October peak, with a company that [beat revenue estimates four quarters running](#6-what-are-you-buying) underneath it.

<img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/0/02/Jensen_Huang_-_Nvidia_Keynote_-_CES_2025_Las_Vegas_%281%29.jpg/960px-Jensen_Huang_-_Nvidia_Keynote_-_CES_2025_Las_Vegas_%281%29.jpg" alt="Jensen Huang on stage at CES 2025 in front of a slide showing AI's path from perception AI to physical AI" width="100%">

*Huang at CES in January 2025, laying out Nvidia's roadmap from generative AI to robots. Investors now have to price that roadmap without him. Photo: [Pronoia](https://commons.wikimedia.org/wiki/File:Jensen_Huang_-_Nvidia_Keynote_-_CES_2025_Las_Vegas_(1).jpg), [CC0](https://creativecommons.org/publicdomain/zero/1.0/), via Wikimedia Commons.*

So: is this a discount on a great business, or the start of a longer slide? The numbers below come straight from the [Financial Modeling Prep](https://site.financialmodelingprep.com/) API and answer it in steps:

1. **The fall**: how far, and when.
2. **Nvidia or the market?** A regression model says what the stock *should* have done.
3. **"Oversold" means buy?** Moving averages, Bollinger Bands, and RSI, tested on 11 years of data.
4. **How bad was Thursday?**
5. **What does a forecast see?** Meta's Prophet, tuned and cross-validated.
6. **What are you buying?** The business underneath.
7. **What did Wall Street do?**
8. **Buy or wait?**

Download the notebook or open it in Colab with the buttons above and run every step yourself.

## Setup

```bash
pip install pandas numpy plotly requests scikit-learn prophet
```

A free FMP key covers everything here. Set it as the `FMP_API_KEY` environment variable, or paste it when asked.

In [1]:
import logging
import os
import warnings
from getpass import getpass

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from plotly.subplots import make_subplots
from prophet import Prophet
from prophet.diagnostics import cross_validation
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Prophet logs every fit; keep the output to results
for name in ["cmdstanpy", "prophet"]:
    logging.getLogger(name).disabled = True
# Prophet's uncertainty ranges come from random sampling; a fixed seed makes reruns match
np.random.seed(7)

# the key stays out of the notebook: read it from the environment, or ask for it
API_KEY = os.environ.get("FMP_API_KEY") or getpass("FMP API key: ")
BASE = "https://financialmodelingprep.com/stable/"

def fmp(endpoint, **params):
    """Call one FMP endpoint and return the JSON."""
    response = requests.get(BASE + endpoint, params={**params, "apikey": API_KEY}, timeout=30)
    response.raise_for_status()
    return response.json()

# one look for every chart: dark, transparent, and colorblind-friendly
COLORS = {"NVDA": "#34d399", "SPY": "#94a3b8", "AMD": "#fb923c", "TSM": "#a78bfa", "MSFT": "#38bdf8", "Down": "#f87171", "Hold": "#94a3b8"}
pio.templates["post"] = go.layout.Template(
    layout=dict(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Inter, system-ui, sans-serif", color="#cbd5e1", size=13),
        title=dict(font=dict(size=17, color="#f1f5f9"), x=0, xanchor="left"),
        xaxis=dict(gridcolor="rgba(148,163,184,0.15)", zerolinecolor="rgba(148,163,184,0.3)"),
        yaxis=dict(gridcolor="rgba(148,163,184,0.15)", zerolinecolor="rgba(148,163,184,0.3)"),
        colorway=list(COLORS.values()),
        margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
    )
)
pio.templates.default = "plotly_dark+post"
# show charts in JupyterLab and VS Code, and keep their data in the saved notebook
pio.renderers.default = "plotly_mimetype+notebook_connected"

## 1. The fall

The news came in pieces, mostly while the market was closed ([CNBC](https://www.cnbc.com/2026/11/12/nvidia-nvda-jensen-huang-second-affair.html), [NBC News](https://www.nbcnews.com/business/markets/nvidia-stock-huang-resigns-rcna608958)):

| Date | What happened |
| --- | --- |
| Sat, Nov 7 | Financial Times reports a relationship with a businesswoman tied to a cloud company that does business with Nvidia |
| Sun, Nov 8 | Huang admits he was unfaithful to his wife |
| Wed, Nov 11 | New York Times reports a relationship with an Nvidia executive who reports to him. Lawmakers call for him to go |
| Thu, Nov 12 | Reuters reports a second employee. Huang resigns. Kress named interim CEO; former Synopsys CEO Harvey C. Jones becomes the first board chair |
| Tue, Nov 17 | Third-quarter earnings, the first under an interim CEO |

In [2]:
nvda = pd.DataFrame(fmp("historical-price-eod/full", symbol="NVDA", **{"from": "2015-01-01"}))
nvda["date"] = pd.to_datetime(nvda["date"])
nvda = nvda.set_index("date").sort_index()[["open", "high", "low", "close", "volume"]]
quote = fmp("quote", symbol="NVDA")[0]

close = nvda["close"]
peak = close.loc["2025-11-12":].idxmax()
print(f"Close ${quote['price']:.2f} ({quote['changePercentage']:+.2f}%), market value ${quote['marketCap'] / 1e12:.2f} trillion")
print(f"Peak ${close[peak]:.2f} on {peak:%b %d}. Since then {close.iloc[-1] / close[peak] - 1:.0%}. "
      f"News week (Nov 6 to Nov 12 close): {close.iloc[-1] / close['2026-11-06'] - 1:.1%}")

recent = nvda.loc["2026-08-01":].reset_index()
price_fig = go.Figure(go.Scatter(
    x=recent["date"], y=recent["close"], mode="lines", line=dict(color=COLORS["NVDA"], width=2.5),
    error_y=dict(type="data", symmetric=False, array=recent["high"] - recent["close"],
                 arrayminus=recent["close"] - recent["low"], color="rgba(52,211,153,0.35)", thickness=1.5, width=0),
    hovertemplate="%{x|%b %d}: closed $%{y:.2f}<extra></extra>",
))
for day, label, y in [(peak, "Peak", 1.0), ("2026-11-09", "Weekend reports", 0.82), ("2026-11-12", "Huang resigns", 0.64)]:
    price_fig.add_vline(x=pd.Timestamp(day).timestamp() * 1000, line_dash="dot", line_color="#f1f5f9")
    price_fig.add_annotation(x=day, y=y, yref="paper", text=label, showarrow=False, xanchor="right", xshift=-4, font=dict(color="#f1f5f9"))
price_fig.update_layout(title="Already sliding, then the news hit", yaxis_title="Share price (bars show each day's range)",
                        yaxis_tickprefix="$", xaxis_title="", height=430, showlegend=False)
price_fig.show()

Close $166.19 (-4.34%), market value $4.01 trillion
Peak $239.24 on Oct 06. Since then -31%. News week (Nov 6 to Nov 12 close): -11.0%


The slide started on October 6, a month before any of this was public, and took about a fifth off the price. The news week took another 11%. A buyer needs to know which part is the market and which part is Nvidia.

## 2. Nvidia or the market?

Predict what Nvidia *would* have done from what the market did, then measure the gap. That gap, the **abnormal return**, is the classic **event study** used in finance research and securities lawsuits. It is a supervised learning problem like any other:

| Role | Variable |
| --- | --- |
| Dependent (*y*) | Nvidia's daily return |
| Independent (*X*) | S&P 500 daily return (the SPY fund) |
| Candidate *X* | Daily returns of Microsoft, Apple, Alphabet, Meta, Amazon: big tech, and Nvidia's biggest customers |

Chip stocks like AMD are left out on purpose: they react to Nvidia's news, so they would leak the event into its own baseline.

Splits run in time order, never shuffled, so the model can't learn from the future:

| Split | Dates | Job |
| --- | --- | --- |
| Train | Jan 2023 to Jun 2026 | Fit the models |
| Validation | Jul to Oct 2026 | Pick a model; measure a normal miss |
| Test | Nov 2 to 12, 2026 | Measure the gap during the news |

In [3]:
CUSTOMERS = ["MSFT", "AAPL", "GOOGL", "META", "AMZN"]

def closes(symbol, start="2022-12-01"):
    data = pd.DataFrame(fmp("historical-price-eod/light", symbol=symbol, **{"from": start}))
    return data.assign(date=pd.to_datetime(data["date"])).set_index("date")["price"]

prices = pd.DataFrame({t: closes(t) for t in ["NVDA", "SPY", *CUSTOMERS, "AMD", "TSM"]}).sort_index().dropna()
returns = prices.pct_change().dropna().loc["2023-01-01":]
train, valid, test = returns.loc[:"2026-06-30"], returns.loc["2026-07-01":"2026-10-30"], returns.loc["2026-11-02":]
print(f"Train {len(train)} days, validation {len(valid)}, test {len(test)}")

TARGET = "NVDA"
candidates = {
    "Naive: 0% every day": None,
    "Linear, market only": (LinearRegression(), ["SPY"]),
    "Linear, market + big tech": (LinearRegression(), ["SPY", *CUSTOMERS]),
    "Ridge, market + big tech": (RidgeCV(alphas=np.logspace(-6, -1, 20)), ["SPY", *CUSTOMERS]),
}
scores = []
for name, spec in candidates.items():
    if spec is None:
        p_train, p_valid = np.zeros(len(train)), np.zeros(len(valid))
    else:
        model, cols = spec
        model.fit(train[cols], train[TARGET])
        p_train, p_valid = model.predict(train[cols]), model.predict(valid[cols])
    scores.append({"model": name, "train R²": r2_score(train[TARGET], p_train),
                   "validation R²": r2_score(valid[TARGET], p_valid),
                   "validation RMSE": np.sqrt(mean_squared_error(valid[TARGET], p_valid))})
pd.DataFrame(scores).set_index("model").style.format({"train R²": "{:.2f}", "validation R²": "{:.2f}", "validation RMSE": "{:.2%}"})

Train 875 days, validation 86, test 9


,train R²,validation R²,validation RMSE
model,,,
Naive: 0% every day,-0.01,-0.00,2.32%
"Linear, market only",0.41,0.35,1.86%
"Linear, market + big tech",0.44,0.18,2.09%
"Ridge, market + big tech",0.44,0.18,2.10%


Textbook overfitting: adding big tech fits the training years better and predicts the validation months *worse*. Ridge doesn't save it. The market-only model wins, gets refit on all the calm data, and faces the test days.

In [4]:
calm = pd.concat([train, valid])
market_model = LinearRegression().fit(calm[["SPY"]], calm[TARGET])
normal_miss = (valid[TARGET] - LinearRegression().fit(train[["SPY"]], train[TARGET]).predict(valid[["SPY"]])).std()

result = pd.DataFrame({"actual": test[TARGET], "predicted": market_model.predict(test[["SPY"]])})
result["abnormal"] = result["actual"] - result["predicted"]
news = result.loc["2026-11-09":]
car = news["abnormal"].sum()
t_stat = car / (normal_miss * np.sqrt(len(news)))
print(f"Beta {market_model.coef_[0]:.2f}: Nvidia usually moves about {market_model.coef_[0]:.1f}x the market. Normal daily miss: {normal_miss:.1%}")
print(f"Nov 9-12: model expected {(1 + news['predicted']).prod() - 1:+.1%}, actual {(1 + news['actual']).prod() - 1:+.1%}. "
      f"Abnormal return {car:+.1%} (t = {t_stat:.1f})")

path = pd.DataFrame({"Actual": (1 + result.loc["2026-11-06":, "actual"]).cumprod() - 1,
                     "Expected from the market": (1 + result.loc["2026-11-06":, "predicted"]).cumprod() - 1})
path.iloc[0] = 0
band = 1.96 * normal_miss * np.sqrt(np.arange(len(path)))
exp_ = path["Expected from the market"]
model_fig = go.Figure([
    go.Scatter(x=list(path.index) + list(path.index[::-1]), y=list(exp_ + band) + list((exp_ - band)[::-1]),
               mode="lines", fill="toself", fillcolor="rgba(148,163,184,0.18)", line=dict(width=0), hoverinfo="skip", name="Normal range (95%)"),
    go.Scatter(x=path.index, y=exp_, mode="lines+markers", name="Expected from the market", line=dict(color=COLORS["SPY"], dash="dash"),
               hovertemplate="%{x|%b %d}: %{y:+.1%}<extra></extra>"),
    go.Scatter(x=path.index, y=path["Actual"], mode="lines+markers", name="Nvidia, actual", line=dict(color=COLORS["Down"], width=3),
               hovertemplate="%{x|%b %d}: %{y:+.1%}<extra></extra>"),
])
model_fig.add_hline(y=0, line_dash="dot", line_color="#f1f5f9")
model_fig.update_layout(title="The market said up 7%. Nvidia went down 11%.", yaxis_title="Return since Fri, Nov 6",
                        yaxis_tickformat="+.0%", xaxis_title="", height=430, xaxis=dict(tickformat="%a %b %d", dtick=86400000),
                        legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
model_fig.show()

(1 + returns.loc["2026-11-09":, ["NVDA", "AMD", "TSM", "SPY", *CUSTOMERS]]).prod().sub(1).sort_values() \
    .rename("Return, Nov 9-12").to_frame().style.format("{:+.1%}")

Beta 2.05: Nvidia usually moves about 2.1x the market. Normal daily miss: 1.9%
Nov 9-12: model expected +6.8%, actual -11.0%. Abnormal return -18.0% (t = -4.8)


,"Return, Nov 9-12"
NVDA,-11.0%
MSFT,-1.4%
META,+2.2%
SPY,+3.0%
AMZN,+3.1%
AAPL,+3.7%
GOOGL,+4.9%
TSM,+8.0%
AMD,+30.3%


The market rose 3% from Monday to Thursday. At Nvidia's usual sensitivity, about 2x the market, that predicts a gain near 7%. It lost 11%. The gap, **about -18%**, is nearly five times a normal miss, far beyond chance.

Meanwhile AMD jumped 30% and TSMC 8%. Money didn't leave AI. It left Nvidia, and some went next door.

### Was it the scandal?

The model says the drop was specific to Nvidia. It can't say *which* Nvidia news caused it. The day-by-day gaps help:

In [5]:
news_days = {
    "2026-11-04": "No scandal news yet",
    "2026-11-09": "First trading day after the weekend FT report and Huang's admission",
    "2026-11-10": "",
    "2026-11-11": "NYT reports a relationship with an employee",
    "2026-11-12": "Huang resigns",
}
daily = result.loc[list(news_days)].assign(news=list(news_days.values()))
daily.style.format_index(lambda d: f"{d:%a %b %d}").format("{:+.1%}", subset=["actual", "predicted", "abnormal"])

,actual,predicted,abnormal,news
date,,,,
Wed Nov 04,-3.2%,+3.6%,-6.7%,No scandal news yet
Mon Nov 09,-4.3%,+5.0%,-9.3%,First trading day after the weekend FT report and Huang's admission
Tue Nov 10,-5.5%,-0.6%,-4.9%,
Wed Nov 11,+2.9%,+2.9%,-0.0%,NYT reports a relationship with an employee
Thu Nov 12,-4.3%,-0.6%,-3.8%,Huang resigns


The timing points at the scandal. The biggest gap, about -9%, came Monday, the first trading day after the weekend reports. But Nvidia had already trailed the market by about 7% on November 4, before anything was public, and the same week brought reports on China chip restrictions and weak guidance from AI cloud company Nebius. The 18 points can't all be pinned on one cause.

The link is also indirect. Investors don't price an affair. They price what follows from it: a founder gone with no successor, questions about board oversight after a relationship with a direct report, and, through the contact U.S. officials reportedly view as a security risk, higher odds of tighter China export rules. That last one is a business risk, not a personal one.

**For a buyer, that cuts both ways.** The discount is real and specific to Nvidia, not a market selloff that drags everything down. But it is specific to Nvidia because the risk is: leadership, governance, and China.

## 3. "Oversold" means buy?

Chart readers would call Nvidia a screaming buy right now. Here are the indicators they use:

| Indicator | How it's built | The reading people use |
| --- | --- | --- |
| **SMA 50** | Average of the last 50 closes | The long trend |
| **WMA 20** | Weighted average of 20 closes, newest weighted 20, oldest 1 | The short trend, faster |
| **Bollinger Bands** | 20-day average ± 2 standard deviations. **%B** places the close inside them, 0 to 1 | Below the lower band = stretched |
| **RSI 14** | Average gain vs. average loss over 14 days, 0 to 100 | Under 30 = oversold |
| **Lags** | Returns 1, 2, 3, 5, and 10 days back | Recent momentum |

In [6]:
def wma(series, window):
    """Weighted moving average: the newest day counts `window` times, the oldest once."""
    weights = np.arange(1, window + 1)
    return series.rolling(window).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

def rsi(series, window=14):
    """Relative Strength Index with Wilder's smoothing."""
    change = series.diff()
    gain = change.clip(lower=0).ewm(alpha=1 / window, adjust=False).mean()
    loss = (-change.clip(upper=0)).ewm(alpha=1 / window, adjust=False).mean()
    return 100 - 100 / (1 + gain / loss)

ind = nvda.copy()
ind["return"] = ind["close"].pct_change()
ind["sma50"] = ind["close"].rolling(50).mean()
ind["wma20"] = wma(ind["close"], 20)
mid, sd = ind["close"].rolling(20).mean(), ind["close"].rolling(20).std()
ind["bb_upper"], ind["bb_lower"] = mid + 2 * sd, mid - 2 * sd
ind["pct_b"] = (ind["close"] - ind["bb_lower"]) / (ind["bb_upper"] - ind["bb_lower"])
ind["bb_width"] = (ind["bb_upper"] - ind["bb_lower"]) / mid
ind["rsi14"] = rsi(ind["close"])
for lag in [1, 2, 3, 5, 10]:
    ind[f"lag_{lag}"] = ind["return"].shift(lag - 1)  # lag_1 is today's return, known at today's close

view = ind.loc["2026-06-01":]
tech_fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.06)
hover = "%{x|%b %d}: $%{y:.2f}<extra></extra>"
tech_fig.add_trace(go.Scatter(x=view.index, y=view["bb_upper"], line=dict(width=0), hoverinfo="skip", showlegend=False), row=1, col=1)
tech_fig.add_trace(go.Scatter(x=view.index, y=view["bb_lower"], line=dict(width=0), fill="tonexty", fillcolor="rgba(148,163,184,0.15)",
                              name="Bollinger Bands", hovertemplate=hover), row=1, col=1)
tech_fig.add_trace(go.Scatter(x=view.index, y=view["sma50"], name="SMA 50", line=dict(color=COLORS["TSM"], width=1.5), hovertemplate=hover), row=1, col=1)
tech_fig.add_trace(go.Scatter(x=view.index, y=view["wma20"], name="WMA 20", line=dict(color=COLORS["AMD"], width=1.5), hovertemplate=hover), row=1, col=1)
tech_fig.add_trace(go.Scatter(x=view.index, y=view["close"], name="Close", line=dict(color=COLORS["NVDA"], width=2.5), hovertemplate=hover), row=1, col=1)
tech_fig.add_trace(go.Scatter(x=view.index, y=view["rsi14"], name="RSI 14", line=dict(color=COLORS["MSFT"], width=1.5),
                              hovertemplate="%{x|%b %d}: RSI %{y:.0f}<extra></extra>"), row=2, col=1)
for level in [30, 70]:
    tech_fig.add_hline(y=level, line_dash="dot", line_color="rgba(241,245,249,0.5)", row=2, col=1)
tech_fig.update_yaxes(tickprefix="$", row=1, col=1)
tech_fig.update_yaxes(range=[0, 100], title_text="RSI", row=2, col=1)
tech_fig.update_layout(title="Every indicator reads oversold", height=580, margin=dict(t=100), legend=dict(orientation="h", y=1.03, x=0, yanchor="bottom"))
tech_fig.show()

last = ind.iloc[-1]
print(f"Nov 12: RSI {last['rsi14']:.0f}, %B {last['pct_b']:.2f}, {last['close'] / last['wma20'] - 1:+.0%} vs. WMA 20, "
      f"{last['close'] / last['sma50'] - 1:+.0%} vs. SMA 50")
print("Closes below the lower band since Oct 1: " + ", ".join(f"{d:%b %d}" for d in ind.loc["2026-10-01":].query("close < bb_lower").index))

Nov 12: RSI 26, %B 0.02, -14% vs. WMA 20, -23% vs. SMA 50
Closes below the lower band since Oct 1: Oct 15, Oct 16, Nov 05, Nov 06, Nov 09, Nov 10


Below the lower band, RSI in the 20s, 23% under the 50-day average. By the chart-reading rules, a buy signal.

**But do these signals predict anything?** Test them as features in a classifier:

| Role | Variable |
| --- | --- |
| Dependent (*y*) | Did Nvidia close **up the next day**? (1 or 0) |
| Independent (*X*) | The 5 lags, distance from WMA 20 and SMA 50, %B, band width, RSI, volume vs. its 50-day average, all known at today's close |

| Split | Dates |
| --- | --- |
| Train | 2015 to 2024 |
| Validation | Jan 2025 to Oct 2026 |

The target is tomorrow and every feature is today. Mixing those up is the most common way a backtest lies.

In [7]:
ind["vs_wma20"] = ind["close"] / ind["wma20"] - 1
ind["vs_sma50"] = ind["close"] / ind["sma50"] - 1
ind["volume_ratio"] = ind["volume"] / ind["volume"].rolling(50).mean()
ind["next_return"] = ind["return"].shift(-1)
ind["up_tomorrow"] = (ind["next_return"] > 0).astype(int)

X_COLS = ["lag_1", "lag_2", "lag_3", "lag_5", "lag_10", "vs_wma20", "vs_sma50", "pct_b", "bb_width", "rsi14", "volume_ratio"]
data = ind.dropna(subset=X_COLS).iloc[:-1]  # the last day has no tomorrow yet
c_train, c_valid = data.loc[:"2024-12-31"], data.loc["2025-01-01":"2026-10-30"]

classifiers = {
    "Baseline: always say up": None,
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=7),
}
rows = []
for name, clf in classifiers.items():
    if clf is None:
        pr_train, pr_valid, auc = np.ones(len(c_train)), np.ones(len(c_valid)), 0.5
    else:
        clf.fit(c_train[X_COLS], c_train["up_tomorrow"])
        pr_train, pr_valid = clf.predict_proba(c_train[X_COLS])[:, 1], clf.predict_proba(c_valid[X_COLS])[:, 1]
        auc = roc_auc_score(c_valid["up_tomorrow"], pr_valid)
    rows.append({"model": name, "train accuracy": accuracy_score(c_train["up_tomorrow"], pr_train > 0.5),
                 "validation accuracy": accuracy_score(c_valid["up_tomorrow"], pr_valid > 0.5), "validation AUC": auc})
print(f"Train {len(c_train):,} days, validation {len(c_valid):,} days")
pd.DataFrame(rows).set_index("model").style.format("{:.1%}", subset=["train accuracy", "validation accuracy"]).format("{:.2f}", subset=["validation AUC"])

Train 2,467 days, validation 459 days


,train accuracy,validation accuracy,validation AUC
model,,,
Baseline: always say up,53.3%,52.1%,0.50
Logistic regression,54.4%,51.9%,0.50
Gradient boosting,71.6%,54.7%,0.52


In [8]:
# the direct test: after an "oversold" day, what happened next? every day since 2015
buckets = pd.cut(data["rsi14"], [0, 30, 45, 55, 70, 100], labels=["Under 30 (oversold)", "30-45", "45-55", "55-70", "Over 70 (overbought)"])
data.groupby(buckets, observed=True).agg(days=("up_tomorrow", "size"), up_next_day=("up_tomorrow", "mean"),
                                         avg_next_day=("next_return", "mean")) \
    .style.format({"days": "{:,}", "up_next_day": "{:.0%}", "avg_next_day": "{:+.2%}"})

,days,up_next_day,avg_next_day
rsi14,,,
Under 30 (oversold),45,53%,+0.34%
30-45,555,56%,+0.21%
45-55,774,53%,+0.24%
55-70,"1,126",53%,+0.26%
Over 70 (overbought),434,49%,+0.21%


Nothing beats a coin flip. Gradient boosting scores 72% on the years it trained on and about 55% on days it hasn't seen, barely above "always say up." An AUC around 0.5 means no real ranking skill. After "oversold" days, Nvidia rose the next day about as often as after any other day.

The indicators describe how stretched the price is. They don't say what happens next, and they can't see a CEO resign.

## 4. How bad was Thursday?

Two measures against every day since 2015: how far the price fell from the prior close at its worst, and volume compared with normal.

In [9]:
nvda["worst_point"] = nvda["low"] / nvda["close"].shift(1) - 1
nvda["volume_vs_normal"] = nvda["volume"] / nvda["volume"].rolling(50).mean().shift(1)
thu = nvda.iloc[-1]
print(f"Low {thu['worst_point']:.1%}: only {(nvda['worst_point'] < thu['worst_point']).sum()} days since 2015 went lower. "
      f"Volume {thu['volume_vs_normal']:.1f}x normal. Low to close: +{thu['close'] / thu['low'] - 1:.1%}")

shock = nvda.dropna(subset=["worst_point", "volume_vs_normal"]).copy()
shock["label"] = np.where(shock.index == shock.index[-1], "Nov 12, 2026", "Every other day since 2015")
shock = shock.sort_values("label", ascending=False)
shock_fig = px.scatter(shock.reset_index(), x="worst_point", y="volume_vs_normal", color="label", custom_data=["date"],
                       color_discrete_map={"Nov 12, 2026": COLORS["Down"], "Every other day since 2015": "rgba(148,163,184,0.35)"},
                       title="Thursday ranks with Nvidia's worst days in a decade",
                       labels={"worst_point": "Lowest price of the day vs. prior close", "volume_vs_normal": "Volume vs. 50-day average", "label": ""})
shock_fig.update_traces(marker=dict(size=6), hovertemplate="%{customdata[0]|%b %d, %Y}: low %{x:.1%}, volume %{y:.1f}x<extra></extra>")
shock_fig.update_traces(selector=dict(name="Nov 12, 2026"), marker=dict(size=14, line=dict(color="#f1f5f9", width=1.5)))
for day, text in [("2025-01-27", "DeepSeek, Jan 2025"), ("2020-03-16", "COVID crash, Mar 2020"), ("2018-11-16", "Crypto bust, Nov 2018")]:
    shock_fig.add_annotation(x=nvda.loc[day, "worst_point"], y=nvda.loc[day, "volume_vs_normal"], text=text, showarrow=True,
                             arrowcolor="#94a3b8", font=dict(color="#cbd5e1", size=12), ax=40, ay=-30)
shock_fig.update_layout(xaxis_tickformat=".0%", yaxis_ticksuffix="x", height=460, legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
shock_fig.show()

Low -14.3%: only 6 days since 2015 went lower. Volume 5.7x normal. Low to close: +11.7%


At its worst, Thursday ranks seventh since 2015, near the 2018 crypto bust, the COVID crash, and DeepSeek, on more volume than any of them. The difference: those days closed near their lows. Thursday closed 12% off its low, as [retail investors bought heavily in the first half hour](https://www.abc.net.au/news/2026-11-13/asx-markets-business-news-live-updates-nov-13/107337368). The market judged the exit serious, not fatal.

## 5. What does a forecast see?

[Prophet](https://facebook.github.io/prophet/), Meta's open-source forecasting library (it started at Facebook), models a price as a trend that can bend at **changepoints**, plus optional seasonality and outside **regressors**, with an uncertainty range.

| Setting | Choice |
| --- | --- |
| Dependent (*y*) | Log of Nvidia's close, so equal percentage moves are equal steps |
| Regressor | Log of the S&P 500 (counterfactual model only) |
| Seasonality | Off. Stocks have no reliable weekly or yearly cycle, and fitting one invents it |
| `changepoint_range` | 0.98, so the trend can bend in recent months (the default, 0.8, freezes the last fifth of history) |
| `changepoint_prior_scale` | How freely the trend bends. **Tuned**, not guessed |

### A counterfactual: Nvidia without the news

Same splits as section 2, tuned on validation by **MAPE**, the average miss as a percentage of price.

In [10]:
logp = np.log(prices.loc["2023-01-01":, ["NVDA", "SPY"]])
frame = pd.DataFrame({"ds": logp.index, "y": logp["NVDA"].values, "spy": logp["SPY"].values})
f_train = frame[frame["ds"] <= "2026-06-30"]
f_valid = frame[(frame["ds"] >= "2026-07-01") & (frame["ds"] <= "2026-10-30")]

def prophet_model(flexibility, with_market=True):
    model = Prophet(changepoint_prior_scale=flexibility, changepoint_range=0.98, interval_width=0.8,
                    daily_seasonality=False, weekly_seasonality=False, yearly_seasonality=False)
    if with_market:
        model.add_regressor("spy")
    return model

def mape(forecast, actual):
    np.random.seed(7)
    return np.mean(np.abs(np.exp(forecast["yhat"].values) / np.exp(actual["y"].values) - 1))

GRID = [0.01, 0.05, 0.2, 0.5]
tuning = pd.DataFrame([{"changepoint_prior_scale": f, "validation MAPE": mape(prophet_model(f).fit(f_train).predict(f_valid), f_valid)}
                       for f in GRID])
best_flex = tuning.loc[tuning["validation MAPE"].idxmin(), "changepoint_prior_scale"]

counterfactual = prophet_model(best_flex).fit(pd.concat([f_train, f_valid]))
history = frame[frame["ds"] >= "2026-08-01"]
np.random.seed(7)
cf = counterfactual.predict(history)
for col in ["yhat", "yhat_lower", "yhat_upper"]:
    cf[col] = np.exp(cf[col])
cf["actual"] = np.exp(history["y"].values)
end = cf.iloc[-1]
print(f"Best changepoint_prior_scale {best_flex} (validation MAPE {tuning['validation MAPE'].min():.1%})")
print(f"Nov 12: actual ${end['actual']:.2f}, Prophet given the market ${end['yhat']:.2f} (80% range ${end['yhat_lower']:.2f} to ${end['yhat_upper']:.2f})")

cf_fig = go.Figure([
    go.Scatter(x=list(cf["ds"]) + list(cf["ds"][::-1]), y=list(cf["yhat_upper"]) + list(cf["yhat_lower"][::-1]),
               mode="lines", fill="toself", fillcolor="rgba(148,163,184,0.18)", line=dict(width=0), hoverinfo="skip", name="Prophet 80% range"),
    go.Scatter(x=cf["ds"], y=cf["yhat"], name="Prophet, given the market", line=dict(color=COLORS["SPY"], dash="dash"),
               hovertemplate="%{x|%b %d}: $%{y:.2f}<extra></extra>"),
    go.Scatter(x=cf["ds"], y=cf["actual"], name="Nvidia, actual", line=dict(color=COLORS["NVDA"], width=2.5),
               hovertemplate="%{x|%b %d}: $%{y:.2f}<extra></extra>"),
])
cf_fig.add_vrect(x0="2026-11-01", x1="2026-11-13", fillcolor="rgba(248,113,113,0.08)", line_width=0, annotation_text="Test", annotation_position="top left")
cf_fig.update_layout(title="Where the market alone would have left Nvidia", yaxis_tickprefix="$", xaxis_title="", height=440,
                     legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
cf_fig.show()

Best changepoint_prior_scale 0.01 (validation MAPE 3.9%)
Nov 12: actual $166.19, Prophet given the market $207.50 (80% range $193.67 to $222.98)


Given what the market actually did, Prophet puts Thursday's close near **$208**, with an 80% range of $194 to $223. The real close, $166, sits far below the bottom of that range. Two different models, one on daily returns and one on the price, agree: the market alone does not explain November. (Prophet's gap, about 20%, is a little wider than the regression's 18% because it covers all of November, including a sharp drop on November 4, before any news was public.)

### Looking ahead

For the future, Prophet can't use the S&P 500 (nobody knows its future either), so this model uses the price alone. Tuning uses Prophet's own **rolling cross-validation**: train on two years, forecast the next 35 trading days, slide forward a month, repeat. That gives 22 honest out-of-sample tests per setting, and a check on whether the "80%" range really holds the price 80% of the time.

In [11]:
history_all = frame[["ds", "y"]]
cv_rows, outlooks = [], {}
for flex in GRID:
    model = prophet_model(flex, with_market=False).fit(history_all)
    cv = cross_validation(model, initial="730 days", period="30 days", horizon="35 days", disable_tqdm=True)
    cv_rows.append({"changepoint_prior_scale": flex, "tests": cv["cutoff"].nunique(),
                    "MAPE": np.mean(np.abs(np.exp(cv["yhat"]) / np.exp(cv["y"]) - 1)),
                    "80% range held the price": ((cv["y"] >= cv["yhat_lower"]) & (cv["y"] <= cv["yhat_upper"])).mean()})
    future = model.make_future_dataframe(periods=35, freq="B", include_history=False)
    np.random.seed(7)
    outlooks[flex] = model.predict(future[future["ds"] <= "2026-12-31"])
cv_table = pd.DataFrame(cv_rows).set_index("changepoint_prior_scale")
cv_table["Dec 31 forecast"] = [np.exp(outlooks[f]["yhat"].iloc[-1]) for f in GRID]
chosen = cv_table["MAPE"].idxmin()
cv_table.style.format_index("{:g}").format({"MAPE": "{:.1%}", "80% range held the price": "{:.0%}", "Dec 31 forecast": "${:.0f}"})

,tests,MAPE,80% range held the price,Dec 31 forecast
changepoint_prior_scale,,,,
0.01,22,11.9%,57%,$232
0.05,22,7.5%,65%,$190
0.2,22,8.1%,59%,$122
0.5,22,9.0%,55%,$116


In [12]:
ahead = outlooks[chosen].copy()
for col in ["yhat", "yhat_lower", "yhat_upper"]:
    ahead[col] = np.exp(ahead[col])
day = lambda d: ahead[ahead["ds"] == d].iloc[0]
print(f"Chosen setting {chosen}. Nov 17: ${day('2026-11-17')['yhat']:.0f} (80% range ${day('2026-11-17')['yhat_lower']:.0f} to ${day('2026-11-17')['yhat_upper']:.0f}). "
      f"Dec 31: ${ahead['yhat'].iloc[-1]:.0f} (${ahead['yhat_lower'].iloc[-1]:.0f} to ${ahead['yhat_upper'].iloc[-1]:.0f})")

actual_recent = nvda.loc["2026-08-01":, "close"]
ahead_fig = go.Figure([
    go.Scatter(x=list(ahead["ds"]) + list(ahead["ds"][::-1]), y=list(ahead["yhat_upper"]) + list(ahead["yhat_lower"][::-1]),
               mode="lines", fill="toself", fillcolor="rgba(52,211,153,0.15)", line=dict(width=0), hoverinfo="skip", name="80% range"),
    go.Scatter(x=ahead["ds"], y=ahead["yhat"], name="Forecast", line=dict(color=COLORS["NVDA"], dash="dash"),
               hovertemplate="%{x|%b %d}: $%{y:.0f}<extra></extra>"),
    go.Scatter(x=actual_recent.index, y=actual_recent, name="Actual", line=dict(color=COLORS["NVDA"], width=2.5),
               hovertemplate="%{x|%b %d}: $%{y:.2f}<extra></extra>"),
])
ahead_fig.add_vline(x=pd.Timestamp("2026-11-17").timestamp() * 1000, line_dash="dot", line_color="#f1f5f9")
ahead_fig.add_annotation(x="2026-11-17", y=1.0, yref="paper", text="Earnings", showarrow=False, xanchor="left", xshift=4, font=dict(color="#f1f5f9"))
ahead_fig.update_layout(title="Prophet's best guess, and how little it knows", yaxis_tickprefix="$", xaxis_title="", height=430,
                        legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
ahead_fig.show()

Chosen setting 0.05. Nov 17: $203 (80% range $188 to $221). Dec 31: $190 ($171 to $209)


**The setting matters more than the model.** The four settings put Nvidia anywhere from $116 to $232 on December 31. Cross-validation picks 0.05, the smallest average miss at 7.5%, and even then its "80%" range held the real price only 65% of the time. The true uncertainty is wider than the shaded band.

The forecast also lags. Its range for Tuesday, $188 to $221, doesn't include today's price: a gently bending trend hasn't caught up with a two-week drop, which is why the center line sits above $166. Prophet knows nothing about earnings, the CEO search, or China. Read it for its width, not its center: the stock could plausibly finish the year far higher or far lower, and the model can't say which.

## 6. What are you buying?

A CEO's exit changes who decides. It doesn't change last quarter's sales.

<img src="https://upload.wikimedia.org/wikipedia/commons/b/b2/Colette_Kress_%281%29.png" alt="Portrait of Colette Kress" width="220" style="width:220px;max-width:60%;border-radius:12px">

*Colette Kress, Nvidia's CFO since 2013 and now interim CEO, will present Tuesday's results. Photo: SiliconANGLE theCUBE, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Colette_Kress_(1).png).*

In [13]:
income = pd.DataFrame(fmp("income-statement", symbol="NVDA", period="quarter", limit=5))
income["quarter"] = income["period"] + " FY" + income["fiscalYear"].astype(str).str[-2:]
income = income.iloc[::-1].set_index("quarter")[["revenue", "grossProfit", "netIncome"]] / 1e9
earnings = pd.DataFrame(fmp("earnings", symbol="NVDA", limit=5))
upcoming = earnings[earnings["revenueActual"].isna()].iloc[0]
reported = earnings.dropna(subset=["revenueActual"])
ratios = fmp("ratios-ttm", symbol="NVDA")[0]

print(f"Revenue, latest quarter vs. a year earlier: {income['revenue'].iloc[-1] / income['revenue'].iloc[0] - 1:+.0%}")
print("Revenue vs. estimates, last four reports: " + ", ".join(f"{b:+.1%}" for b in reported["revenueActual"] / reported["revenueEstimated"] - 1))
print(f"Net margin {ratios['netProfitMarginTTM']:.0%}. Price/earnings {ratios['priceToEarningsRatioTTM']:.1f}x. "
      f"Nov 17 estimate: ${upcoming['revenueEstimated'] / 1e9:.1f}B revenue, ${upcoming['epsEstimated']:.2f} EPS")

bars = income[["revenue"]].assign(kind="Reported")
bars.loc["Q3 FY27 (est.)"] = [upcoming["revenueEstimated"] / 1e9, "Analyst estimate"]
rev_fig = px.bar(bars.reset_index(), x="quarter", y="revenue", color="kind", text="revenue",
                 color_discrete_map={"Reported": COLORS["NVDA"], "Analyst estimate": COLORS["SPY"]},
                 title="Revenue doubled in a year", labels={"quarter": "", "revenue": "Quarterly revenue, billions", "kind": ""})
rev_fig.update_traces(texttemplate="$%{text:.0f}B", textposition="outside", hovertemplate="%{x}: $%{y:.1f} billion<extra></extra>")
rev_fig.update_layout(yaxis_range=[0, 125], height=400, legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
rev_fig.show()

Revenue, latest quarter vs. a year earlier: +106%
Revenue vs. estimates, last four reports: +4.3%, +3.4%, +3.0%, +3.7%
Net margin 64%. Price/earnings 20.9x. Nov 17 estimate: $108.7B revenue, $2.47 EPS


Revenue up 106% in a year. Four straight beats. Sixty-four cents of profit on every dollar of sales. At $166 the stock trades near 21 times trailing earnings, a modest price for that growth, because the price fell while profits rose.

<img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/4/42/Jensen_Huang_-_RTX_Blackwell_-_Nvidia_Keynote_-_CES_2025_Las_Vegas_%282%29.jpg/960px-Jensen_Huang_-_RTX_Blackwell_-_Nvidia_Keynote_-_CES_2025_Las_Vegas_%282%29.jpg" alt="A Blackwell chip shown on a giant screen at CES 2025, with specifications listed on both sides" width="100%">

*Blackwell, the chip generation behind the recent growth, at CES 2025: 92 billion transistors. The product line doesn't leave with the CEO. Photo: [Pronoia](https://commons.wikimedia.org/wiki/File:Jensen_Huang_-_RTX_Blackwell_-_Nvidia_Keynote_-_CES_2025_Las_Vegas_(2).jpg), [CC0](https://creativecommons.org/publicdomain/zero/1.0/), via Wikimedia Commons.*

## 7. What did Wall Street do?

In [14]:
grades = pd.DataFrame(fmp("grades", symbol="NVDA"))
grades["date"] = pd.to_datetime(grades["date"])
november = grades[grades["date"] >= "2026-11-01"].copy()
november["action"] = november["action"].map({"downgrade": "Downgrade", "maintain": "Kept rating", "upgrade": "Upgrade"})
counts = november.groupby(["date", "action"]).size().rename("firms").reset_index()
grade_fig = px.bar(counts, x="date", y="firms", color="action",
                   color_discrete_map={"Downgrade": COLORS["Down"], "Kept rating": COLORS["Hold"], "Upgrade": COLORS["NVDA"]},
                   title="Nine downgrades in two days. Zero sells.", labels={"date": "", "firms": "Rating actions", "action": ""})
grade_fig.update_traces(hovertemplate="%{x|%b %d}: %{y} firms<extra></extra>")
grade_fig.update_layout(height=380, xaxis_dtick=86400000, xaxis_tickformat="%b %d", legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
grade_fig.show()

target = fmp("price-target-consensus", symbol="NVDA")[0]
downgrades = november[november["action"] == "Downgrade"]["gradingCompany"]
print(f"Downgrades: {', '.join(downgrades)}")
print(f"Consensus target ${target['targetConsensus']:.0f} (low ${target['targetLow']:.0f}, high ${target['targetHigh']:.0f}) vs. close ${quote['price']:.2f}")

Downgrades: Oppenheimer, StoneX, William Blair, Keybanc, Bernstein, Evercore ISI Group, Melius Research, Craig-Hallum, Baird
Consensus target $328 (low $200, high $515) vs. close $166.19


Nine firms cut Nvidia from buy to hold. None said sell, and ten, including Goldman Sachs and Bank of America, kept their ratings Thursday. The consensus target is about twice the price, but many targets predate the news. The low end is [Bernstein, which halved its target to $200](https://www.cnbc.com/2026/11/12/nvidia-nvda-jensen-huang-second-affair.html), still 20% above Thursday's close.

## 8. Buy or wait?

### The case for buying

- **The business is intact.** Revenue doubled in a year, margins sit above 60%, and the company has beaten estimates four quarters running.
- **The price is lower than the profits.** About 21 times earnings, after a 31% fall from the peak.
- **The market already flinched and recovered.** Thursday closed 12% off its low. Even the most bearish target on the board, $200, sits above today's price.

### The case for waiting

- **The discount is about Nvidia, not the market.** Over the news week Nvidia trailed what the market predicted by about 18 points, a gap specific to Nvidia rather than the market. That is a risk premium for real unknowns: no permanent CEO, and the China questions raised by the reports, which JPMorgan analysts said [had raised the odds of tighter chip restrictions](https://www.nbcnews.com/business/markets/nvidia-stock-huang-resigns-rcna608958).
- **The trend was down before the news.** The slide began October 6. "Oversold" readings have not predicted bounces in 11 years of Nvidia data.
- **Tuesday decides a lot.** Earnings on November 17 are the first test under an interim CEO, and no model above can see them.

### What the data can and can't say

The data can say the drop is real, specific to Nvidia, and not about the business itself. It can't say who runs Nvidia next, what Washington does, or what Tuesday's numbers bring. Those are judgment calls. Anyone buying at $166 is betting on them, not on a chart.

*This is analysis of what happened, not advice to buy or sell anything. Only put in money you can afford to lose.*

### Limits

- The event study assumes Nvidia's link to the market over the past year still holds.
- The November 9 to 12 window also had other news, including disappointing guidance from AI cloud company Nebius. Not all of the abnormal return is the CEO.
- FMP's free plan covers daily data only; intraday prices and news need a paid plan.

### Sources

1. CNBC, [Nvidia stock closes down over 4% after CEO Jensen Huang announces resignation](https://www.cnbc.com/2026/11/12/nvidia-nvda-jensen-huang-second-affair.html), Nov 12, 2026.
2. NBC News, [Nvidia stock falls after Huang resigns, weighing on broader market](https://www.nbcnews.com/business/markets/nvidia-stock-huang-resigns-rcna608958), Nov 12, 2026.
3. ABC News (Australia), [Nvidia shares dive after Jensen Huang's exit](https://www.abc.net.au/news/2026-11-13/asx-markets-business-news-live-updates-nov-13/107337368), Nov 13, 2026.
4. Financial Modeling Prep, [stable API](https://site.financialmodelingprep.com/developer/docs): prices, quote, income statements, earnings calendar, ratios, analyst grades, and price targets, pulled Nov 13, 2026.
5. Taylor, S. J., and Letham, B. (2018). [Forecasting at scale](https://peerj.com/preprints/3190/). *The American Statistician*, 72(1). The Prophet paper.
6. MacKinlay, A. C. (1997). Event Studies in Economics and Finance. *Journal of Economic Literature*, 35(1), 13-39.
7. Bollinger, J. (2001). *Bollinger on Bollinger Bands*. McGraw-Hill. Wilder, J. W. (1978). *New Concepts in Technical Trading Systems*, the source of RSI.